In [51]:
%matplotlib tk
import scipy as sc
import numpy as np
import sympy as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(['science','notebook', 'grid'])

In [52]:
N = 250

t, g = sp.symbols('t g')
m1, m2 = sp.symbols('m1 m2')
L1, L2 = sp.symbols('L1 L2')

Define $\theta_{1}$ and $\theta_{2}$ to be functions of time

In [53]:
the1, the2 = sp.symbols(r'\theta_1, \theta_2',cls=sp.Function)

the1 = the1(t)
the2 = the2(t)

Define the angular derivatives

In [54]:
the1_d = sp.diff(the1, t)
the2_d = sp.diff(the2, t)
the1_dd = sp.diff(the1_d, t)
the2_dd = sp.diff(the2_d, t)

Parametrize our x and y coordinates

In [55]:
x1 = L1 * sp.sin(the1)
y1 = -L1 * sp.cos(the1)
x2 = L1 * sp.sin(the1) + L2 * sp.sin(the2)
y2 = -L1 * sp.cos(the1) - L2 * sp.cos(the2)

To define our Lagrangian L we must define our kinetic and potential energy

In [56]:
# Kinetic Energy Term
T1 = 0.5 * m1 * (sp.diff(x1,t)**2+sp.diff(y1,t)**2)
T2 = 0.5 * m2 * (sp.diff(x2,t)**2+sp.diff(y2,t)**2)
T = T1 + T2

# Potential Energy Term
U1 = m1 * g * y1
U2 = m2 * g * y2
U = U1 + U2

# Lagrangian
L = T - U

Get Lagranges equations of motion
$$ \frac{\partial L}{\partial \theta_{1}}-\frac{d}{dt}\frac{\partial L}{\partial \dot{\theta_{1}}} = 0$$
$$ \frac{\partial L}{\partial \theta_{2}}-\frac{d}{dt}\frac{\partial L}{\partial \dot{\theta_{2}}} = 0$$

In [57]:
LE1 = sp.diff(L,the1) - sp.diff(sp.diff(L,the1_d),t).simplify()
LE2 = sp.diff(L,the2) - sp.diff(sp.diff(L,the2_d),t).simplify()
LE1

-L1*g*m1*sin(\theta_1(t)) - L1*g*m2*sin(\theta_1(t)) - 1.0*L1*(L1*m1*Derivative(\theta_1(t), (t, 2)) + L1*m2*Derivative(\theta_1(t), (t, 2)) - L2*m2*sin(\theta_1(t) - \theta_2(t))*Derivative(\theta_1(t), t)*Derivative(\theta_2(t), t) + L2*m2*sin(\theta_1(t) - \theta_2(t))*Derivative(\theta_2(t), t)**2 + L2*m2*cos(\theta_1(t) - \theta_2(t))*Derivative(\theta_2(t), (t, 2))) + 0.5*m2*(2*L1*(L1*sin(\theta_1(t))*Derivative(\theta_1(t), t) + L2*sin(\theta_2(t))*Derivative(\theta_2(t), t))*cos(\theta_1(t))*Derivative(\theta_1(t), t) - 2*L1*(L1*cos(\theta_1(t))*Derivative(\theta_1(t), t) + L2*cos(\theta_2(t))*Derivative(\theta_2(t), t))*sin(\theta_1(t))*Derivative(\theta_1(t), t))

Solve for $\ddot{\theta_{1}}$ and $\ddot{\theta_{2}}$ in terms of everything else

In [ ]:
sols = sp.solve([LE1, LE2], (the1_dd, the2_dd), simplify = False, rational=False)


Use sp.lambdify to turn these symbolic equations into numerical functions

In [42]:
dw_1dt_f = sp.lambdify((t,g,m1,m2,L1,L2,the1,the2,the1_d,the2_d), sols[the1_dd])
dw_2dt_f = sp.lambdify((t,g,m1,m2,L1,L2,the1,the2,the1_d,the2_d), sols[the2_dd])
dthe1dt_f = sp.lambdify(the1_d,the1_d)
dthe2dt_f = sp.lambdify(the2_d,the2_d)

This gives us a system of first order linear differential equations that can be solved in scipy with odeint. We define $\vec{S} = (\theta_{1}, \omega_{1}, \theta_{2}, \omega_{2})$

In [43]:
def dSdt(S, t ,g, m1, m2, L1, L2):
    the1, w1, the2, w2 = S
    return [
        dthe1dt_f(w1),
        dw_1dt_f(t, g, m1, m2, L1, L2, the1, the2, w1, w2),
        dthe2dt_f(w2),
        dw_2dt_f(t, g, m1, m2, L1, L2, the1, the2, w1, w2)
    ]

In [44]:
t = np.linspace(0, 10, N)
g = 9.81
m1 = 2
m2 = 1
L1 = 2
L2 = 1
ans = sc.integrate.odeint(dSdt, y0=[1, -1, 1, 4], t=t, args=(g,m1,m2,L1,L2))

In [45]:
the1 = ans.T[0]
the2 = ans.T[2]
plt.plot(t,the1)
plt.plot(t,the2)

In [46]:
def get_x1y1x2y2(t, the1,the2,L1,L2):
    return (
        L1 * np.sin(the1),
        -L1 * np.cos(the1),
        L1 * np.sin(the1) + L2 * np.sin(the2),
        -L1 * np.cos(the1) - L2 * np.cos(the2)
    )

x1, y1, x2, y2 = get_x1y1x2y2(t, ans.T[0], ans.T[2], L1, L2)

In [47]:
def animate(i):
    ln1.set_data([0, x1[i], x2[i]], [0, y1[i], y2[i]])

In [48]:
from matplotlib import pyplot as plt, animation

fig, ax = plt.subplots()
ax.set_facecolor('k')
ax.set(xlim=(-4, 4), ylim=(-4, 4))
ln1, = plt.plot([], [], 'ro--', markersize=8)

ani = animation.FuncAnimation(fig, animate, frames = N-1, interval = 50)
#ani.save(filename="/Users/hasan/Python Animations/Double Pendulum Non-Chaotic.gif", writer="pillow")
#plt.show(ani)